# Exploration and Exploitation

1. Introduction
2. Multi-Armed Bandits
3. Contextual Bandits
4. MDPs


## Exploration vs. Exploitation Dilemma

- Online decision-making involves a fundamental choice:
    - **Exploitation** Make the best decision given current information
    - **Exploration** Gather more information
- The best long-term strategy may involve short-term sacrifices
- Gather enough information to make the best overall decisions

### Examples:

- Restaurant Selection
    - Exploitation Go to your favourite restaurant
    - Exploration Try a new restaurant
- Online Banner Advertisements
    - Exploitation Show the most successful advert
    - Exploration Show a diﬀerent advert
- Oil Drilling
    - Exploitation Drill at the best known location
    - Exploration Drill at a new location
- Game Playing
    - Exploitation Play the move you believe is best
    - Exploration Play an experimental move


### Principles

#### Naive Exploration
- Add noise to greedy policy (e.g. ε-greedy)
- The simplest approach: occasionally pick a random action instead of the best known one, so the agent stumbles upon unexplored options.

#### Optimistic Initialisation
- Assume the best until proven otherwise
- Start with inflated value estimates for all actions. The agent will naturally try under-explored actions first because they still look promising on paper.

#### Optimism in the Face of Uncertainty
- Prefer actions with uncertain values
- Actively seek out actions where the value estimate has high uncertainty (e.g. UCB). Uncertainty is treated as a bonus — unknown = potentially great.

#### Probability Matching
- Select actions according to the probability they are best
- Sample an action proportional to how likely it is to be optimal (e.g. Thompson Sampling). Balances exploration and exploitation probabilistically.

#### Information State Search
- Lookahead search incorporating value of information
- Plan ahead by treating the *knowledge gained* from an action as part of its reward. Explicitly reasons about which actions are most informative.

## The Multi-Armed Bandit

- A multi-armed bandit is a tuple $\langle \mathcal{A}, \mathcal{R} \rangle$
- $\mathcal{A}$ is a known set of $m$ actions (or "arms")
- $\mathcal{R}^a(r) = \mathbb{P}[r|a]$ is an unknown probability distribution over rewards
- At each step $t$ the agent selects an action $a_t \in \mathcal{A}$
- The environment generates a reward $r_t \sim \mathcal{R}^{a_t}$
- The goal is to maximise cumulative reward $\sum_{\tau=1}^{t} r_\tau$

<img src="imgs/image-124.png" width=200px>

### Simple Explanation

Think of a row of slot machines ("one-armed bandits"), each with a different — and unknown — payout rate.

- **Arms**: the $m$ machines you can pull.
- **$\mathcal{R}^a$**: each machine has its own hidden reward distribution; you don't know which is best.
- **At each step**: you pick one machine to pull.
- **Reward**: you observe a random payout sampled from that machine's distribution.
- **Goal**: maximise total winnings over time.

The core challenge is the **explore/exploit trade-off** — do you keep pulling the machine that has paid out best so far, or try others that might be even better?

### Regret

- The **action-value** is the mean reward for action $a$:
$$Q(a) = \mathbb{E}[r|a]$$

- The **optimal value** $V^*$ is:
$$V^* = Q(a^*) = \max_{a \in \mathcal{A}} Q(a)$$

- The **regret** is the opportunity loss for **one step**:
$$l_t = \mathbb{E}[V^* - Q(a_t)]$$

- The **total regret** is the total opportunity loss:
$$L_t = \mathbb{E}\left[\sum_{\tau=1}^{t} V^* - Q(a_\tau)\right]$$

- Maximise cumulative reward $\equiv$ minimise total regret

#### Simple Explanation

- **$Q(a)$**: the average reward you expect from pulling arm $a$.
- **$V^*$**: the average reward of the *best possible* arm — the ceiling you're aiming for.
- **Regret at step $t$**: how much worse your chosen action $a_t$ is compared to always picking the best arm. Zero regret means you picked optimally.
- **Total regret $L_t$**: the accumulated gap over all steps — how much reward you *missed out on* by not always picking the best arm.

Minimising total regret and maximising total reward are exactly the same objective — you want your choices to be as close to the optimal arm as possible, as often as possible.

### Counting Regret

- The **count** $N_t(a)$ is the expected number of selections for action $a$
- The **gap** $\Delta_a$ is the difference in value between action $a$ and optimal action $a^*$:
$$\Delta_a = V^* - Q(a)$$
- Regret is a function of gaps and counts:

$$L_t = \mathbb{E}\left[\sum_{\tau=1}^{t} V^* - Q(a_\tau)\right]$$
$$= \sum_{a \in \mathcal{A}} \mathbb{E}[N_t(a)](V^* - Q(a))$$
$$= \sum_{a \in \mathcal{A}} \mathbb{E}[N_t(a)]\, \Delta_a$$

- A good algorithm ensures small counts for large gaps
- **Problem: gaps are not known!**

#### Simple Explanation

Total regret decomposes neatly: for each arm $a$, multiply **how often you pulled it** by **how suboptimal it was**, then sum across all arms.

$$L_t = \sum_{a} \underbrace{\mathbb{E}[N_t(a)]}_{\text{how often}} \times \underbrace{\Delta_a}_{\text{how bad}}$$

- A large gap $\Delta_a$ means arm $a$ is much worse than optimal — you want to pull it rarely.
- A small gap means the arm is nearly as good as optimal — pulling it occasionally is not costly.

The ideal algorithm quickly identifies bad arms and stops pulling them. The catch: you never directly observe $\Delta_a$, so you have to *infer* which arms are bad from noisy reward samples.

### Linear or Sublinear Regret

<img src=imgs/image-125.png width=500px />

- If an algorithm **forever** explores it will have linear total regret
- If an algorithm **never** explores it will have linear total regret
- Is it possible to achieve sublinear total regret?

#### Simple Explanation

The graph shows total regret over time for three strategies:

- **Greedy** (blue): never explores — locks in on whatever looked best early on, which may be suboptimal. Regret grows linearly because it keeps making the same mistake forever.
- **ε-greedy** (red): always explores at fixed rate — better than pure greedy, but still linear because it keeps wasting a fixed fraction of steps on random actions even after learning enough.
- **Decaying ε-greedy** (black): explores a lot early, then less and less over time — regret curve bends and flattens, achieving **sublinear** growth.

**Key insight**: both extremes (always explore, never explore) are equally bad in the long run — linear regret. The goal is an algorithm that explores *just enough* early on and gradually commits to the best arm, so that total regret grows sublinearly (e.g. $O(\log t)$).

### Greedy Algorithm

- We consider algorithms that estimate $\hat{Q}_t(a) \approx Q(a)$
- Estimate the value of each action by Monte-Carlo evaluation:

$$\hat{Q}_t(a) = \frac{1}{N_t(a)} \sum_{t=1}^{T} r_t \mathbf{1}(a_t = a)$$

- The **greedy** algorithm selects the action with highest estimated value:

$$a_t^* = \underset{a \in \mathcal{A}}{\operatorname{argmax}}\, \hat{Q}_t(a)$$

- Greedy can lock onto a suboptimal action forever
- $\Rightarrow$ Greedy has linear total regret

#### Simple Explanation

- **$\hat{Q}_t(a)$**: running average of all rewards received when arm $a$ was pulled. The indicator $\mathbf{1}(a_t = a)$ just means "only count timesteps where we actually chose $a$".
- **Greedy selection**: always pull the arm with the highest average reward so far — pure exploitation, zero exploration.

**Why it fails**: if by chance a suboptimal arm looks good early on (lucky initial pulls), greedy commits to it and never tries the others enough to discover the true best arm. That early mistake compounds forever → linear regret.

### ε-Greedy Algorithm

The **ε-greedy** algorithm always keeps exploring — it never fully stops trying random actions.

**Action selection rule:**

- With probability **1 − ε**: exploit — pick the action with the highest estimated value:
  `a = argmax Q̂(a)` over all `a ∈ A`
- With probability **ε**: explore — pick a **random action**

**Regret bound:**

Because ε is constant, at every step there is always a fixed chance of picking a suboptimal action.
This means the per-step regret never goes to zero, giving a **lower bound on cumulative regret**:

$$
l_t \geq \frac{\varepsilon}{\mathcal{A}} \sum_{a \in \mathcal{A}} \Delta_a
$$

where Δ_a is the **gap** between the optimal action's value and action `a`'s value.

**Consequence:** constant ε → **linear total regret** — regret grows proportionally with time, which is suboptimal compared to algorithms that decay exploration over time.

### Optimistic Initialisation

**Core idea:** initialise Q(a) to an artificially high value for all actions, then update using incremental Monte-Carlo.

**Requirement:** N(a) > 0 for all actions (each action must be "seen" at least once at the start).

**Update rule:**

$$
\hat{Q}_t(a_t) = \hat{Q}_{t-1} + \frac{1}{N_t(a_t)}(r_t - \hat{Q}_{t-1})
$$

This is a running average: the new estimate moves toward the observed reward $r_t$ by a step size of $1/N(a_t)$.

**Why it encourages exploration:**

By initialising Q(a) high, every action looks promising at the start.
Actions that get selected quickly disappoint (real rewards are lower), so the agent naturally tries other actions — systematic early exploration without an explicit ε.

**Limitation:**

The agent can still **lock onto a suboptimal action** once estimates converge, because there is no forced exploration mechanism after the initial phase.

**Regret:**

- Greedy + optimistic initialisation → **linear total regret**
- ε-greedy + optimistic initialisation → **linear total regret**

Optimistic initialisation improves early exploration but does **not** fix the fundamental regret scaling — both combinations still grow linearly over time.

## Decaying εt-Greedy Algorithm

Instead of a fixed ε, use a **schedule** ε₁, ε₂, ... that decreases over time.

**A concrete schedule:**

- c > 0 — a tunable constant
- $d = \underset{a | Δ_a > 0}{min} Δ_i$ — the **smallest gap** among suboptimal actions (how close the best suboptimal action is to the optimal)
- The decay schedule:

$$
\varepsilon_t = \min\left\{1,\ \frac{c|\mathcal{A}|}{d^2 t}\right\}
$$

Exploration probability shrinks as `1/t` — early on ε ≈ 1 (explore a lot), later ε → 0 (exploit more and more).

**Result:** decaying εt-greedy achieves **logarithmic asymptotic total regret** — a major improvement over linear regret.

**The catch:**

The schedule depends on `d`, the minimum gap between actions.
This requires **advance knowledge of the reward gaps** Δ_a, which is typically unknown in practice.

**Motivation for what comes next:**

Goal: find an algorithm that achieves **sublinear regret** for any multi-armed bandit problem, **without** knowing the reward distribution R or the gaps.


### UCB (Upper Confidence Bound)

UCB  is the standard answer to that goal.

Instead of using gaps explicitly, UCB estimates **uncertainty** for each action and adds an exploration bonus:


#### Lower Bound

**Key insight:** how hard a bandit problem is depends on how similar the suboptimal arms look to the optimal arm.

- If a suboptimal arm has a reward distribution very close to the optimal, it is hard to distinguish them → more exploration needed → more regret
- This similarity is measured by the **KL divergence** $KL(R^a || R^{a*})$: how different arm `a`'s distribution is from the optimal arm `a*`

**Theorem (Lai and Robbins):**

Asymptotic total regret is **at least** logarithmic in the number of steps:

$$
\lim_{t \to \infty} L_t \geq \log t \sum_{a | \Delta_a > 0} \frac{\Delta_a}{KL(\mathcal{R}^a || \mathcal{R}^{a^*})}
$$

**Reading the bound:**

- Large Δ_a (suboptimal arm is clearly bad) but also large KL (easy to distinguish) → small contribution to regret
- Small Δ_a and small KL (arm looks almost like the optimal) → large contribution → hard problem

**Implication:**

No algorithm can do better than logarithmic regret in general. Logarithmic growth is therefore the **gold standard** — an algorithm that achieves it (like UCB) is asymptotically optimal.

#### Uncertainty and Exploration

<img src=imgs/image-126.png width=600px>

The plot shows posterior distributions p(Q) over the value of three actions a₁, a₂, a₃.

- **Q(a₁)** (blue): wide, flat distribution — very uncertain, mean looks low
- **Q(a₂)** (red): moderate uncertainty, medium mean
- **Q(a₃)** (green): narrow, peaked distribution — high confidence, highest mean

**Which action should we pick?**

The greedy choice is a₃ (highest estimated value). But a₁ has enormous uncertainty — its true value **could** be much higher than it currently appears.

**The principle:**

> The more uncertain we are about an action's value, the more important it is to explore it — it could turn out to be the best action.

This is the intuition behind UCB: don't just pick the action with the highest estimated value, pick the one with the highest **upper confidence bound**, rewarding uncertainty as a reason to explore.

<img src=imgs/image-127.png width=600px>

- After picking the **blue** action, we are less uncertain about its value
- This makes us more likely to pick another action next
- The process repeats until we home in on the best action

#### Upper Confidence Bounds

**Core idea:** for each action, maintain an upper confidence bound $Û_t(a)$ such that the true value Q(a) lies below the estimated value plus the bound with high probability:

$$
Q(a) \leq \hat{Q}_t(a) + \hat{U}_t(a)
$$

**The bound depends on how many times action `a` has been selected, N_t(a):**

- Small $N_t(a)$ → large $ Û_t(a)$: the estimate is uncertain, so the upper bound is wide
- Large $N_t(a)$ → small $Û_t(a)$: the estimate is accurate, so the upper bound is tight

**Action selection — pick the action with the highest UCB:**

$$
a_t = \underset{a \in \mathcal{A}}{\argmax}\ \hat{Q}_t(a) + \hat{U}_t(a)
$$

**Intuition:** this automatically balances exploration and exploitation:
- A well-explored action wins if its **estimated value** is high
- An under-explored action wins if its **uncertainty bonus** is high
- Once an action is pulled more, its bonus shrinks and it must compete on value alone

#### Hoeffding's Inequality

**Theorem (Hoeffding's Inequality):**

Let $X₁, ..., X_t$ be i.i.d. random variables in [0,1], and let $\bar{X}_t = \frac{1}{t} \sum_{\tau=1}^{t} X_\tau$ be the sample mean. Then:

$$
\mathbb{P}\left[\mathbb{E}[X] > \bar{X}_t + u\right] \leq e^{-2tu^2}
$$

**Plain English:** the probability that the true mean exceeds the sample mean by more than `u` drops exponentially as you collect more samples `t` or increase the margin `u`. This is valid for any distribution.

**Applying this to the bandit:**

Conditioned on selecting action `a`, treat rewards as i.i.d. samples. Then:

$$
\mathbb{P}\left[Q(a) > \hat{Q}_t(a) + U_t(a)\right] \leq e^{-2N_t(a)U_t(a)^2}
$$

- Q(a) is the true value (the true mean)
- Q̂_t(a) is the sample mean after N_t(a) pulls
- U_t(a) is the confidence width we want to set

**Key use:** we can choose U_t(a) to make this probability as small as we like — this tells us **how wide the confidence bound needs to be** to guarantee Q(a) ≤ Q̂_t(a) + U_t(a) with high probability.

#### Calculating Upper Confidence Bounds

**Step 1:** pick a probability `p` that the true value exceeds the UCB, then solve for $U_t(a)$.

From Hoeffding's inequality, set the bound equal to `p`:

$$
e^{-2N_t(a)U_t(a)^2} = p
$$

Solving for $U_t(a)$:

$$
U_t(a) = \sqrt{\frac{-\log p}{2N_t(a)}}
$$

The bound shrinks as $N_t(a)$ grows — the more we pull action `a`, the tighter the confidence interval.

**Step 2:** reduce `p` over time so we keep selecting the optimal action as t → ∞.

A natural choice: `p = t⁻⁴`, which shrinks fast enough to ensure convergence. Substituting:

$$
U_t(a) = \sqrt{\frac{2 \log t}{N_t(a)}}
$$

**Intuition:**
- $\log t$ grows slowly → the bound keeps widening slightly over time, ensuring under-explored actions stay attractive
- $N_t(a)$ in the denominator → the more an action is pulled, the tighter its bound becomes

This gives the **UCB1** confidence bound, which achieves logarithmic total regret.


##### UCB1

**Action selection rule:**

$$
a_t = \underset{a \in \mathcal{A}}{\argmax}\ Q(a) + \sqrt{\frac{2 \log t}{N_t(a)}}
$$

At each step, pick the action with the highest estimated value plus an exploration bonus that grows with total time `t` and shrinks with how often action `a` has been pulled.

**Theorem:**

The UCB1 algorithm achieves **logarithmic asymptotic total regret**:

$$
\lim_{t \to \infty} L_t \leq 8 \log t \sum_{a | \Delta_a > 0} \Delta_a
$$

**What this means:**

- Regret is bounded by $\log t$ times the sum of gaps Δ_a over all suboptimal actions
- This matches the **Lai-Robbins lower bound** up to a constant factor — UCB1 is asymptotically optimal
- Crucially, this is achieved **without any knowledge of the reward distributions** R or the gaps $Δ_a$